# Triagent: QLoRA severity classifier (free Colab T4)

Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with QLoRA to classify incident severity under
POL-SEV-001. It then compares it with the **same model, zero-shot, same prompt** on 200
held-out incidents.

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Step 2 asks for `triagent_finetune.zip`, which `python -m finetune.make_bundle` builds in
the repo.

Takes about 30–40 minutes on a free T4: baseline about 3 min, training about 20–25 min,
evaluation about 3 min. Nothing here costs money.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, rerun."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Upload the bundle (`triagent_finetune.zip`)

In [ ]:
import os
import zipfile

from google.colab import files

if not os.path.exists("finetune/data/train.jsonl"):
    uploaded = files.upload()
    name = next(n for n in uploaded if n.endswith(".zip"))
    zipfile.ZipFile(name).extractall(".")
!ls finetune finetune/data && head -c 1500 finetune/data/stats.json

## 3. Install the pinned fine-tuning stack

In [ ]:
!pip -q install -r finetune/requirements.txt

## 4. Baseline: the base model, zero-shot

In [ ]:
!python -m finetune.evaluate --out finetune/results/base.json

## 5. Train QLoRA adapters (4-bit NF4 base, LoRA r=16, completion-only loss)

In [ ]:
!python -m finetune.train_qlora

## 6. Evaluate the fine-tuned model on the same test set

In [ ]:
!python -m finetune.evaluate --adapter finetune/outputs/adapter --out finetune/results/qlora.json

## 7. Compare and download the results

In [ ]:
from IPython.display import Markdown, display

R = "finetune/results"
META = "finetune/outputs/adapter/training_meta.json"
!python -m finetune.evaluate compare {R}/base.json {R}/qlora.json --training {META}
display(Markdown(open(f"{R}/README.md").read()))
!cp {META} {R}/
!cd finetune && zip -qr ../triagent_finetune_results.zip results
files.download("triagent_finetune_results.zip")

Put `triagent_finetune_results.zip` back in the repo and unzip it into `finetune/`, which
creates `finetune/results/`. To keep the adapter (about 70 MB), also download
`finetune/outputs/adapter/` from the file browser on the left.